# Safety Abstractions Pacman Coins

This tutorial shows why safety abstractions are an efficiency tool, not just a compatibility trick. We use `mini_pacman_with_coins` for a fast runnable example; `pacman_with_coins` uses the same idea at a larger scale.

The matching static docs page is at `docs/Tutorials/Shielding/Safety Abstractions Pacman Coins.md`.

## CPU-first setup

Keep the notebook portable and quiet before importing MASA/JAX modules.

In [ ]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

## Imports

The coin environment returns a structured observation with reward-relevant coin bits. The abstraction helpers are already provided by the environment module.

In [ ]:
import numpy as np
from IPython.display import Markdown, display

from masa.common.utils import make_env
from masa.envs.discrete import mini_pacman_with_coins as coins
from masa.prob_shield.prob_shield_wrapper_v1 import ProbShieldWrapperDisc


def markdown_table(rows, columns):
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = ["| " + " | ".join(str(row[column]) for column in columns) + " |" for row in rows]
    return "\n".join([header, divider, *body])

## Build the structured-observation task

`mini_pacman_with_coins` has the same safety abstraction pattern as `pacman_with_coins`, but the smaller state model keeps this notebook quick. The task observation includes coin locations because reward depends on coins; ghost-collision safety does not need to remember every coin bit.

In [ ]:
def make_coins_pctl_env(max_episode_steps=100):
    return make_env(
        "mini_pacman_with_coins",
        "pctl",
        max_episode_steps,
        label_fn=coins.label_fn,
        cost_fn=coins.cost_fn,
        alpha=0.01,
    )


base_env = make_coins_pctl_env()
obs, info = base_env.reset(seed=0)

print("observation_space:", base_env.observation_space)
print("action_space:", base_env.action_space)
print("obs shape:", obs.shape)
print("labels:", info["labels"])
print("constraint:", info["constraint"])

assert obs.shape == (7, 10, 9)
base_env.close()

## Inspect observation channels

The first channel stores coins. The next four channels store the agent direction, and the final four store the ghost direction. Safety only needs the agent and ghost positions/directions, not every coin bit.

In [ ]:
base_env = make_coins_pctl_env()
obs, _ = base_env.reset(seed=0)

coin_channel = obs[:, :, 0]
agent_channels = obs[:, :, 1:5]
ghost_channels = obs[:, :, 5:9]

agent_y, agent_x, agent_direction = np.argwhere(agent_channels == 1.0)[0]
ghost_y, ghost_x, ghost_direction = np.argwhere(ghost_channels == 1.0)[0]

print("coin channel shape:", coin_channel.shape)
print("coins still available:", int(np.sum(coin_channel)))
print("agent y/x/direction:", (int(agent_y), int(agent_x), int(agent_direction)))
print("ghost y/x/direction:", (int(ghost_y), int(ghost_x), int(ghost_direction)))

assert agent_channels.shape == (7, 10, 4)
assert ghost_channels.shape == (7, 10, 4)
base_env.close()

## Compare original and abstract state counts

The important win is state-space compression. A naive original task state would need to distinguish the safety state plus every possible coin mask. The safety abstraction ignores coin masks and keeps only the agent/ghost dynamics needed for ghost-collision safety.

In [ ]:
coin_bits = int(np.prod(coin_channel.shape))
safety_abstract_states = coins.N_STATES
original_state_upper_bound = safety_abstract_states * (2**coin_bits)
compression_factor = original_state_upper_bound // safety_abstract_states

rows = [
    {
        "model": "naive original task state upper bound",
        "states": f"{original_state_upper_bound:.3e}",
        "what it remembers": f"agent/ghost safety state plus {coin_bits} coin bits",
    },
    {
        "model": "safety abstract states",
        "states": f"{safety_abstract_states:,}",
        "what it remembers": "agent and ghost positions/directions only",
    },
]

display(Markdown(markdown_table(rows, ["model", "states", "what it remembers"])))
print("compression factor:", f"{compression_factor:.3e}x fewer states for safety analysis")

assert safety_abstract_states == coins.N_STATES
assert original_state_upper_bound > safety_abstract_states
assert compression_factor == 2**coin_bits

## Inspect the safety abstraction

`safety_abstraction(obs)` extracts the agent and ghost safety state from the structured observation. `abstr_label_fn(abstract_state)` labels that compact state for safety checking, while `cost_fn` keeps the usual ghost-collision cost.

In [ ]:
base_env = make_coins_pctl_env()
obs, _ = base_env.reset(seed=0)

abstract_state = coins.safety_abstraction(obs)
abstract_tuple = coins.REVERSE_STATE_MAP[abstract_state]
abstract_labels = coins.abstr_label_fn(abstract_state)

print("abstract state:", abstract_state)
print("decoded abstract state:", abstract_tuple)
print("abstr_label_fn:", abstract_labels)
print("cost_fn:", coins.cost_fn(abstract_labels))

assert isinstance(int(abstract_state), int)
assert coins.cost_fn(abstract_labels) == 0.0
base_env.close()

## Coin-channel changes do not change safety state

Coin observations matter for reward, but ghost-collision safety depends only on the agent and ghost. If we alter only a coin bit, the raw observation changes while the safety abstraction remains the same.

In [ ]:
base_env = make_coins_pctl_env()
obs, _ = base_env.reset(seed=0)

coin_changed_obs = obs.copy()
coin_changed_obs[0, 0, 0] = 1.0 - coin_changed_obs[0, 0, 0]

same_abstract_state = coins.safety_abstraction(obs) == coins.safety_abstraction(coin_changed_obs)
raw_observations_differ = not np.array_equal(obs, coin_changed_obs)

print("raw observations differ:", raw_observations_differ)
print("abstract states match:", same_abstract_state)

assert raw_observations_differ
assert same_abstract_state
base_env.close()

## Build the shield with the abstraction

Now the shield can compute bounds over the compact discrete safety model while still returning the original structured observation to the learning algorithm.

In [ ]:
def make_shielded_coins_env():
    env = make_coins_pctl_env()
    return ProbShieldWrapperDisc(
        env,
        label_fn=coins.abstr_label_fn,
        cost_fn=coins.cost_fn,
        safety_abstraction=coins.safety_abstraction,
        init_safety_bound=0.01,
        theta=1e-12,
        max_vi_steps=2_000,
        granularity=10,
    )


shielded_env = make_shielded_coins_env()
obs, info = shielded_env.reset(seed=0)

print("shielded observation_space:", shielded_env.observation_space)
print("shielded action_space:", shielded_env.action_space)
print("orig_obs shape:", obs["orig_obs"].shape)
print("safety_bound:", obs["safety_bound"])
print("current abstract state:", shielded_env._current_obs)
print("safety_lb range:", float(np.min(shielded_env.safety_lb)), float(np.max(shielded_env.safety_lb)))
print("successor_states_matrix shape:", shielded_env.successor_states_matrix.shape)
print("probabilities shape:", shielded_env.probabilities.shape)

assert "orig_obs" in obs
assert "safety_bound" in obs
assert obs["orig_obs"].shape == (7, 10, 9)

## Inspect one projected candidate action

As in Tutorial 09, `_parse_act` and `_project_act` are internal inspection hooks. They are useful here to see that the projection is built over abstract successor states, even though `orig_obs` remains structured.

In [ ]:
ACTION_NAMES = {
    0: "right",
    1: "down",
    2: "left",
    3: "up",
    4: "stay",
}


def make_augmented_action(primary_action, fallback_action=None, beta_level=0):
    fallback_action = primary_action if fallback_action is None else fallback_action
    return np.array(
        [primary_action, fallback_action] + [beta_level] * shielded_env.max_successors,
        dtype=int,
    )


def project_candidate_action(env, action):
    i, j, betas = env._parse_act(action)
    safe_actions, bounds, proj_penalty, margin_penalty = env._project_act(i, j, betas)
    n_successors = np.count_nonzero(env.successor_states_matrix[:, env._current_obs] >= 0)
    return {
        "primary": ACTION_NAMES[int(i)],
        "fallback": ACTION_NAMES[int(j)],
        "safe_actions": np.round(safe_actions, 3).tolist(),
        "bounds": np.round(bounds[:n_successors], 4).tolist(),
        "proj_penalty": round(float(proj_penalty), 4),
        "margin_penalty": round(float(margin_penalty), 4),
    }


candidate_action = make_augmented_action(0, beta_level=0)
projection = project_candidate_action(shielded_env, candidate_action)
projection

## Step once

The reward still comes from the coin task. The safety bookkeeping comes from the PCTL constraint and the shield's updated safety bound.

In [ ]:
next_obs, reward, terminated, truncated, step_info = shielded_env.step(candidate_action)

print("reward:", reward)
print("terminated:", terminated)
print("truncated:", truncated)
print("next orig_obs shape:", next_obs["orig_obs"].shape)
print("next safety_bound:", next_obs["safety_bound"])
print("constraint:", step_info["constraint"])
print("proj_penalty:", step_info["proj_penalty"])
print("margin_penalty:", step_info["margin_penalty"])

assert next_obs["orig_obs"].shape == (7, 10, 9)
assert "constraint" in step_info
assert "proj_penalty" in step_info
assert "margin_penalty" in step_info
assert float(next_obs["safety_bound"]) <= 1.0

## Cleanup

Close the shielded environment when finished.

In [ ]:
shielded_env.close()